# 03 — Model Comparison: SVM vs EEGNet vs ShallowNet

**Purpose:** Compare encoding classifier architectures on the PEERS dataset.
Evaluate using cross-subject validation and compare against the EEG-ITNet
baseline (AUC 0.68).

**Models:**
1. SVM on extracted spectral+ERP features (baseline)
2. EEGNet on raw epochs (compact CNN)
3. ShallowFBCSPNet on raw epochs (temporal-spatial CNN)

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import mne
mne.set_log_level("WARNING")
plt.style.use("dark_background")

from classifiers.encoding.data import (
    PEERSConfig, load_peers_dataset, merge_datasets, split_by_subject
)
from classifiers.encoding.features import EncodingFeatureConfig, extract_encoding_features
from classifiers.encoding.models import create_model
from classifiers.encoding.train import (
    train_and_evaluate, run_cross_validation, prepare_features, EEGITNET_BASELINE
)
from shared.evaluation.metrics import compute_classification_metrics

PEERS_DIR = Path("../../data/raw/peers")

## 1. Load and Split Data

In [ ]:
# Load all subjects
config = PEERSConfig(bids_root=PEERS_DIR)
datasets = load_peers_dataset(config)
dataset = merge_datasets(datasets)

# Split by subject
split = split_by_subject(dataset, config)

print(f"Total subjects: {len(np.unique(dataset.subjects))}")
print(f"Train: {len(split.train_subjects)} subjects, {len(split.train.epochs)} epochs")
print(f"Val: {len(split.val_subjects)} subjects, {len(split.val.epochs)} epochs")
print(f"Test: {len(split.test_subjects)} subjects, {len(split.test.epochs)} epochs")
print(f"\nLabel balance (train): {split.train.labels.mean():.2f} recalled")

## 2. Train All Models and Compare

In [ ]:
# Train and evaluate each model
feature_config = EncodingFeatureConfig()
results = {}

# 1. SVM Baseline
print("=" * 50)
print("Training SVM...")
svm_model, svm_metrics, svm_preds = train_and_evaluate(
    split.train, split.test, model_type="svm", feature_config=feature_config
)
results["SVM"] = svm_metrics
print(f"  AUC: {svm_metrics.auc:.4f}, Acc: {svm_metrics.accuracy:.4f}, F1: {svm_metrics.f1:.4f}")

# 2. EEGNet
print("\nTraining EEGNet...")
try:
    eegnet_model, eegnet_metrics, eegnet_preds = train_and_evaluate(
        split.train, split.test, model_type="eegnet",
        model_kwargs={"n_epochs": 50, "lr": 1e-3}
    )
    results["EEGNet"] = eegnet_metrics
    print(f"  AUC: {eegnet_metrics.auc:.4f}, Acc: {eegnet_metrics.accuracy:.4f}, F1: {eegnet_metrics.f1:.4f}")
except Exception as e:
    print(f"  EEGNet failed: {e}")

# 3. ShallowNet
print("\nTraining ShallowNet...")
try:
    shallow_model, shallow_metrics, shallow_preds = train_and_evaluate(
        split.train, split.test, model_type="shallownet",
        model_kwargs={"n_epochs": 50, "lr": 1e-3}
    )
    results["ShallowNet"] = shallow_metrics
    print(f"  AUC: {shallow_metrics.auc:.4f}, Acc: {shallow_metrics.accuracy:.4f}, F1: {shallow_metrics.f1:.4f}")
except Exception as e:
    print(f"  ShallowNet failed: {e}")

## 3. Results Comparison Table and Visualization

In [ ]:
# Comparison table
print(f"\n{'='*65}")
print(f"{'Model':<15} {'Accuracy':>10} {'AUC':>10} {'F1':>10} {'Precision':>10} {'Recall':>10}")
print(f"{'-'*65}")

# Add baseline
print(f"{'EEG-ITNet*':<15} {EEGITNET_BASELINE['accuracy']:>10.4f} "
      f"{EEGITNET_BASELINE['auc']:>10.4f} {EEGITNET_BASELINE['f1']:>10.4f} {'—':>10} {'—':>10}")
print(f"{'-'*65}")

for name, metrics in results.items():
    print(f"{name:<15} {metrics.accuracy:>10.4f} {metrics.auc:>10.4f} "
          f"{metrics.f1:>10.4f} {metrics.precision:>10.4f} {metrics.recall:>10.4f}")

print(f"{'='*65}")
print("* Published baseline from literature")

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metric_names = ["accuracy", "auc", "f1"]
metric_labels = ["Accuracy", "AUC", "F1 Score"]

for ax, metric, label in zip(axes, metric_names, metric_labels):
    model_names = list(results.keys()) + ["EEG-ITNet\n(baseline)"]
    values = [getattr(results[m], metric) for m in results] + [EEGITNET_BASELINE[metric]]
    colors = ["#4FC3F7"] * len(results) + ["#999"]

    bars = ax.bar(model_names, values, color=colors, edgecolor="#333", linewidth=2)
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_ylim(0, 1)

    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", fontsize=9)

    # Baseline line
    ax.axhline(y=EEGITNET_BASELINE[metric], color="#FFD54F",
               linestyle="--", linewidth=1, alpha=0.7)

plt.suptitle("Encoding Classifier — Model Comparison", fontsize=13)
plt.tight_layout()
plt.show()

# Check if we beat baseline
best_model = max(results.items(), key=lambda x: x[1].auc)
if best_model[1].auc >= EEGITNET_BASELINE["auc"]:
    print(f"\n✓ {best_model[0]} beats EEG-ITNet baseline! "
          f"(AUC {best_model[1].auc:.4f} >= {EEGITNET_BASELINE['auc']:.4f})")
else:
    print(f"\n✗ Best model ({best_model[0]}, AUC={best_model[1].auc:.4f}) "
          f"does not yet beat baseline ({EEGITNET_BASELINE['auc']:.4f})")